In [55]:
%load_ext autoreload
%autoreload 2

import jax
jax.config.update("jax_enable_x64", True)

from mGST.low_level_jit import cost_function_jax_mps
from mGST.utility_functions_comparisons import GSTConfiguration, get_full_mgst_parameters_from_configuration, get_compressed_rep_from_mgst_output, get_isometry_dimensions_from_tensor

from mGST.trust_region import riemannian_hessian_vector_fn, riemannian_gradient_fn, truncated_conjugate_gradient
from mGST.trust_region_modified import linearize_riemannian_gradient, rhessian_vector_product_from_linear_map
from mGST.trust_region_modified import riemannian_gradient_fn as new_riemannian_gradient_fn
from mGST.trust_region_modified import truncated_conjugate_gradient as new_truncated_conjugate_gradient
from mGST.automatic_diff import hvp

import jax.numpy as jnp

backend = "iqmfakeapollo"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Manual check for linearize

In [4]:
# First using the JAX compilation with smaller qubits to see if this helps
dim_1q = 2
num_circuits_1q = 100
shots = 1000
kraus_rank = 4

Q1_GST = GSTConfiguration(
    qubit_layouts=[[0]],
    gate_set="1QXYI",
    num_circuits=num_circuits_1q,
    shots=shots,
    rank=kraus_rank,
)

kraus_tensor_mgst_1q, kraus_mgst_1q, povm_mgst_1q, state_mgst_1q, prob_matrix_1q, indices_list_1q = get_full_mgst_parameters_from_configuration(
    Q1_GST, backend, only_jax_variables=True
)

kraus_tensor_1q_target, povm_psd_1q_target, state_psd_1q_target = get_compressed_rep_from_mgst_output(kraus_mgst_1q, povm_mgst_1q, state_mgst_1q, kraus_rank=kraus_rank, state_rank=dim_1q, povm_rank=dim_1q)


2026-09-10 14:47:28,775 - iqm.benchmarks.logging_config - INFO - Generating 100 random GST circuits
2026-09-10 14:47:29,071 - iqm.benchmarks.logging_config - INFO - Will transpile all 100 circuits according to fixed physical layout
2026-09-10 14:47:29,218 - iqm.benchmarks.logging_config - INFO - Transpiling for backend IQMFakeApolloBackend with optimization level 0, sabre routing method all circuits
2026-09-10 14:47:30,260 - iqm.benchmarks.logging_config - INFO - Submitting batch with 100 circuits corresponding to qubits [0]
2026-09-10 14:47:30,270 - iqm.benchmarks.logging_config - INFO - Stored jobs for 1 layouts in dataset
2026-09-10 14:47:30,289 - iqm.benchmarks.logging_config - INFO - Now executing the corresponding circuit batch
2026-09-10 14:47:30,314 - iqm.benchmarks.logging_config - INFO - Retrieving all counts
2026-09-10 14:47:30,668 - iqm.benchmarks.logging_config - INFO - Adding counts to dataset
2026-09-10 14:47:30,818 - iqm.benchmarks.logging_config - INFO - Run completed


In [10]:
input_1q = {
    "kraus_tensor": kraus_tensor_1q_target,
    "povm_psd": povm_psd_1q_target,
    "state_psd": state_psd_1q_target,
    "prob_matrix": prob_matrix_1q,
    "indices_list": indices_list_1q,
    "jit": True,
}

input_no_kraus = {
    "povm_psd": povm_psd_1q_target,
    "state_psd": state_psd_1q_target,
    "prob_matrix": prob_matrix_1q,
    "indices_list": indices_list_1q,
    "jit": True,
}

In [7]:
%time cost_function_jax_mps(**input_1q)


CPU times: user 410 ms, sys: 25.5 ms, total: 436 ms
Wall time: 429 ms


Array(0.00251687, dtype=float64)

In [12]:
cost_fn_kraus = lambda x: cost_function_jax_mps(x, **input_no_kraus)
operator_type = "kraus"
metric = "euclidean"

In [14]:
rhessian_vector_fn = lambda x, z: riemannian_hessian_vector_fn(x=x, tangent_vector=z, cost_fn=cost_fn_kraus, operator_type=operator_type, metric=metric, return_tensor=True)

x = kraus_tensor_1q_target
z = 2 * kraus_tensor_1q_target

In [15]:
%time rhessian_vector_fn(x, z)

CPU times: user 11.2 s, sys: 377 ms, total: 11.6 s
Wall time: 5.29 s


Array([[[[-2.46384569e-01+2.30311705e-01j,
           1.90324232e+00+2.81550561e-01j],
         [-1.90451347e+00+3.96423494e-01j,
          -1.69089086e-01-2.26128176e-01j]],

        [[ 5.01144359e-01+3.93294588e-02j,
          -1.32699049e-01+1.37315757e-01j],
         [-5.17373001e-01-6.63037186e-01j,
          -5.03723971e-01-7.49439943e-02j]],

        [[ 2.63780100e-01+4.08088987e-04j,
          -2.12693620e-01+7.37437138e-02j],
         [ 5.82231777e-01+7.75912942e-01j,
          -2.18160029e-01+4.97752852e-02j]],

        [[-5.74515792e-02+2.19779111e-02j,
          -2.43741052e-01+2.53922961e-02j],
         [ 6.34603646e-03-1.07723484e-01j,
           8.04684589e-02-3.73981156e-02j]]],


       [[[-2.10764203e-01-2.23635217e-01j,
           8.69631389e-01+3.10536584e-02j],
         [-9.04490287e-01+8.82919709e-02j,
          -2.16183596e-01+2.57484808e-01j]],

        [[ 4.62856034e-01+6.92330088e-02j,
          -2.99965578e-01-4.14965925e-01j],
         [-3.04740970e-01+2.057

In [16]:
time_rhessian_old = %timeit -o rhessian_vector_fn(x, z)

164 ms ± 1.31 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
rgradient, euclidean_gradient, linearized_map = %time linearize_riemannian_gradient(x=x, cost_fn=cost_fn_kraus, operator_type=operator_type, metric=metric)

CPU times: user 6.81 s, sys: 278 ms, total: 7.08 s
Wall time: 3.38 s


In [22]:
rhessian_test = %time rhessian_vector_product_from_linear_map(x=x, delta_tensor=z, rgradient=rgradient, linearized_hvp_map=linearized_map, operator_type=operator_type, metric=metric)

CPU times: user 7.12 s, sys: 299 ms, total: 7.42 s
Wall time: 3.16 s


In [27]:
test_object = linearized_map(z)
type(test_object)

tuple

In [25]:
rhessian_og = rhessian_vector_fn(x, z)
jnp.allclose(rhessian_og, rhessian_test), jnp.linalg.norm(rhessian_og - rhessian_test)

(Array(False, dtype=bool), Array(189.43570603, dtype=float64))

In [38]:
rgrad_fn = lambda x: riemannian_gradient_fn(x=x, cost_fn=cost_fn_kraus, operator_type=operator_type, metric=metric) # Riemannian gradient (from df/dx*)

rgrad_fn_new = lambda x: new_riemannian_gradient_fn(x=x, cost_fn=cost_fn_kraus, operator_type=operator_type, metric=metric) # New Riemannian gradient (from df/dx*)

In [33]:
rgrad_x_tensor_og, Drgrad_x_to_z_tensor_og = hvp(function=rgrad_fn, x=x, z=z)

In [34]:
rgrad_x_tensor_new, hvp_map = jax.linearize(rgrad_fn, x)
Drgrad_x_t_z_tensor_new = hvp_map(z)

In [35]:
jnp.allclose(Drgrad_x_to_z_tensor_og, Drgrad_x_t_z_tensor_new)

Array(True, dtype=bool)

In [40]:
(rgrad_tensor_new_2, euclidean_tensor_new_2), hvp_map_2 = jax.linearize(rgrad_fn_new, x)

output_new_map = hvp_map_2(z)

In [45]:
jnp.allclose(Drgrad_x_to_z_tensor_og, output_new_map[0]), jnp.allclose(Drgrad_x_t_z_tensor_new, output_new_map[1])

(Array(True, dtype=bool), Array(False, dtype=bool))

In [46]:
time_rhessian_old = %timeit -o hvp(function=rgrad_fn, x=x, z=z)
time_rhessian_new = %timeit -o hvp_map(z)

163 ms ± 8.85 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
30.7 ms ± 437 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [47]:
time_rhessian_new_2 = %timeit -o hvp_map_2(z)

31.1 ms ± 486 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [48]:
time_linearize_new = %timeit -o jax.linearize(rgrad_fn, x)
time_linearize_new_2 = %timeit -o jax.linearize(rgrad_fn_new, x)

393 ms ± 5.78 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
390 ms ± 2.19 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [52]:
print(f"{'operator':<8} {'old (best)':>14} {'new (best)':>14} {'difference':>12}")
old_best = time_rhessian_old.best
new_best = time_rhessian_new.best
difference = old_best - new_best
print(f"{operator_type:<8} {old_best * 1e3:>11.3f} ms {new_best * 1e3:>11.3f} ms {difference * 1e3:>9.3f} ms")

operator     old (best)     new (best)   difference
kraus        155.687 ms      30.052 ms   125.635 ms


In [56]:
n, p = get_isometry_dimensions_from_tensor(x, operator_type)

In [61]:
rgradient_new, euclidean_gradient_new, linearized_map_new = linearize_riemannian_gradient(x=x, cost_fn=cost_fn_kraus, operator_type=operator_type, metric=metric)

In [66]:
# now we can test how long the whole optimization takes.
solution_old, on_boundary_old = truncated_conjugate_gradient(x=x, radius=0.1, num_iterations=10, rgradient=rgradient, metric=metric, n=n, p=p, rhessian_vector_fn=rhessian_vector_fn, verbose=True)

TCG finished after: 4/10 iters. 
 Reason: Negative curvature encountered 📉.
---------------------------------------


In [67]:
solution_new, on_boundary_new = new_truncated_conjugate_gradient(x=x, radius=0.1, num_iterations=10, rgradient=rgradient, operator_type=operator_type, metric=metric, n=n, p=p, linearized_hvp_map=linearized_map_new, verbose=True)

TCG finished after: 4/10 iters. 
 Reason: Negative curvature encountered 📉.
---------------------------------------


In [68]:
jnp.allclose(solution_old, solution_new)

Array(True, dtype=bool)

In [70]:
time_tc_old = %timeit -o truncated_conjugate_gradient(x=x, radius=0.1, num_iterations=10, rgradient=rgradient, metric=metric, n=n, p=p, rhessian_vector_fn=rhessian_vector_fn, verbose=False)
time_tc_new = %timeit -o new_truncated_conjugate_gradient(x=x, radius=0.1, num_iterations=10, rgradient=rgradient, operator_type=operator_type, metric=metric, n=n, p=p, linearized_hvp_map=linearized_map_new, verbose=False)

664 ms ± 18.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
135 ms ± 1.36 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [71]:
time_linearization_once = %timeit -o linearize_riemannian_gradient(x=x, cost_fn=cost_fn_kraus, operator_type=operator_type, metric=metric)

382 ms ± 3.85 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
